# Embeddings in Bayesian Models

In [ ]:
from flax import nnx
import jax.numpy as jnp


def mood_init(key, shape, dtype=jnp.float32):
    return jnp.array(
        [
            [1.0, 0.0],  # active
            [-1.0, 0.0],  # chill
        ],
        dtype=dtype,
    )


def weather_init(key, shape, dtype=jnp.float32):
    return jnp.array(
        [
            [0.0, 1.0],  # rainy
            [0.0, -1.0],  # sunny
        ],
        dtype=dtype,
    )


class ContextEmbedding(nnx.Module):
    def __init__(self, *, rngs):
        self.mood_embedding = nnx.Embed(
            num_embeddings=2,
            features=2,
            embedding_init=mood_init,
            rngs=rngs,
        )

        self.weather_embedding = nnx.Embed(
            num_embeddings=2,
            features=2,
            embedding_init=weather_init,
            rngs=rngs,
        )

    def __call__(self, mood, weather):
        return self.mood_embedding(mood) + self.weather_embedding(weather)

In [ ]:
encoder = ContextEmbedding(rngs=nnx.Rngs(0))

mood = jnp.array([0, 0, 1, 1])
weather = jnp.array([0, 1, 0, 1])

z = encoder(mood, weather)

print(z)
# [[ 1.  1.]
#  [ 1. -1.]
#  [-1.  1.]
#  [-1. -1.]]

In [ ]:
import jax
import numpyro
import numpyro.distributions as dist
from jax import random
from numpyro.infer import MCMC, NUTS, Predictive


def model(z, obs=None):
    embedding_dim = z.shape[-1]
    num_activities = 4

    beta_0 = numpyro.sample(
        "beta_0", dist.Normal(0, 1).expand((num_activities,)).to_event(1)
    )

    beta = numpyro.sample(
        "beta", dist.Normal(0, 1).expand((embedding_dim, num_activities)).to_event(2)
    )

    logits = beta_0 + z @ beta

    theta = numpyro.deterministic("theta", jax.nn.softmax(logits, axis=-1))

    numpyro.sample("obs", dist.Multinomial(total_count=25, logits=logits), obs=obs)

In [ ]:
data = jnp.array(
    [
        [0, 0, 15, 10],  # active & rainy
        [15, 5, 0, 5],  # active & sunny
        [0, 0, 5, 20],  # chill & rainy
        [5, 15, 0, 5],  # chill & sunny
    ]
)

In [ ]:
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=2000)
mcmc.run(random.PRNGKey(42), z=z, obs=data)
mcmc.print_summary()
#       mean       std    median      5.0%     95.0%     n_eff     r_hat
#  beta[0,0]      0.41      0.55      0.42     -0.52      1.28    742.39      1.00
#  beta[0,1]     -0.65      0.55     -0.65     -1.56      0.25    765.99      1.00
#  beta[0,2]      0.56      0.56      0.56     -0.39      1.44    656.95      1.00
#  beta[0,3]     -0.27      0.53     -0.27     -1.14      0.57    641.80      1.00
#  beta[1,0]     -1.37      0.65     -1.37     -2.42     -0.28   1216.79      1.00
#  beta[1,1]     -1.44      0.65     -1.45     -2.46     -0.35   1161.71      1.00
#  beta[1,2]      1.94      0.63      1.94      0.94      3.03   1335.41      1.00
#  beta[1,3]      0.87      0.58      0.86     -0.02      1.85   1152.86      1.00
#  beta_0[0]     -0.41      0.65     -0.40     -1.50      0.66   1303.72      1.00
#  beta_0[1]     -0.46      0.64     -0.45     -1.52      0.58    913.07      1.00
#  beta_0[2]     -0.34      0.63     -0.34     -1.36      0.68   1176.22      1.00
#  beta_0[3]      1.24      0.57      1.23      0.32      2.16    915.10      1.00

In [ ]:
samples = mcmc.get_samples()
predictive = Predictive(model, samples, return_sites=["theta"])
pred = predictive(random.PRNGKey(1), z=z)
mean_theta = jnp.mean(pred["theta"], axis=0)

print(mean_theta)

In [ ]:
ACTIVITIES = ["cycling", "picnic", "climbing", "movie"]
MOODS = ["active", "chill"]
WEATHERS = ["rainy", "sunny"]


# Encode one context
mood, weather = 0, 0  # active & rainy
z_query = encoder(jnp.array([mood]), jnp.array([weather]))

# Return posterior samples of preferences for that context
predictive = Predictive(model, samples, return_sites=["theta"])
theta_samples = predictive(random.PRNGKey(1), z=z_query)["theta"][:, 0]
mean_theta = jnp.mean(theta_samples, axis=0)

# Get the index of the activity with the highest mean preference
i = int(jnp.argmax(mean_theta))

print(f"Context: {MOODS[mood]} & {WEATHERS[weather]}")
print(f"Recommended activity: {ACTIVITIES[i]}")
# Context: active & rainy
# Recommended activity: climbing

In [ ]:
encoder(jnp.array([mood]), jnp.array([weather]))